# Prompt Component Pair Response Atlas - reference solution

The task gives, for every training article, the hidden classifier's answer
distribution under all 256 ordered prompt-component pairs. It gives no such
surface for the test articles. So the whole problem is one regression:

    article text + target category  ->  the 256-value response surface

Fit that once, then read every required answer field off the predicted surface
instead of fitting nine separate models. That keeps the fields mutually
consistent, which is what the rubric asks for, and it is also what scores best.

Runs in a couple of minutes. pandas, numpy, scipy and scikit-learn only.

## 1. Imports and configuration

In [ ]:
import json
import os

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge

SEED = 20260803
TOP_K = 5
np.random.seed(SEED)

## 2. Load the public files

`train_atlas.csv` is the important one. It holds 256 rows per training
article - one per ordered pair - with the classifier's probability on each of
the seventeen categories.

In [ ]:
PUB = "./dataset/public"
if not os.path.isfile(os.path.join(PUB, "test.csv")):
    PUB = "./prepared/public"

train = pd.read_csv(os.path.join(PUB, "train.csv"))
test = pd.read_csv(os.path.join(PUB, "test.csv"))
atlas = pd.read_csv(os.path.join(PUB, "train_atlas.csv"))
atlas["article_id"] = atlas["article_id"].astype(str)

print(train.shape, test.shape, atlas.shape)

## 3. Rebuild each training article's measured surface

Normalise each item's 256 values to [0, 1] against that item's own floor and
ceiling. This matters: the grader scores the chosen pair the same way, and raw
probabilities differ by more than an order of magnitude between items, so a
model fitted on raw values would be dominated by a handful of them.

## 4. Featurise, fit one multi-output ridge, and read the fields off

Five small logistic heads cover the fields that describe the classifier rather
than the surface itself.

In [ ]:
here = os.path.dirname(os.path.abspath(__file__))
pub = _pub(here)

train = pd.read_csv(os.path.join(pub, "train.csv"))
test = pd.read_csv(os.path.join(pub, "test.csv"))
atlas = pd.read_csv(os.path.join(pub, "train_atlas.csv"))
atlas["article_id"] = atlas["article_id"].astype(str)

pk0 = json.loads(train.loc[0, "packet_json"])
cats = pk0["categories"]
comps = [c["component_id"] for c in pk0["components"]]
pairs = [(a, b) for a in comps for b in comps]
pair_i = {p: i for i, p in enumerate(pairs)}
slugs = ["p_" + c.split(",")[0].split(" ")[0].lower().replace("-", "_")
         for c in cats]

# ---- measured surfaces for the training articles --------------------- #
surf = {}
for aid, g in atlas.groupby("article_id"):
    m = np.zeros((len(pairs), len(cats)))
    for r in g.itertuples(index=False):
        m[pair_i[(r.comp_a, r.comp_b)]] = [getattr(r, s) for s in slugs]
    surf[aid] = m

def unpack(df):
    rows = []
    for r in df.itertuples(index=False):
        pk = json.loads(r.packet_json)
        rows.append({"id": r.id, "article_id": str(r.article_id),
                     "text": pk["headline"] + " \n " + pk["article"],
                     "target": pk["target_category"]})
    return pd.DataFrame(rows)

tr = unpack(train)
te = unpack(test)

# ---- features -------------------------------------------------------- #
word = TfidfVectorizer(max_features=40000, ngram_range=(1, 2),
                       sublinear_tf=True, min_df=2)
char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                       max_features=40000, sublinear_tf=True, min_df=2)
Xw = word.fit_transform(tr["text"])
Xc = char.fit_transform(tr["text"])
cat_i = {c: i for i, c in enumerate(cats)}

def onehot(series):
    m = np.zeros((len(series), len(cats)))
    for i, c in enumerate(series):
        m[i, cat_i[c]] = 1.0
    return csr_matrix(m)

Xtr = hstack([Xw, Xc, onehot(tr["target"])]).tocsr()
Xte = hstack([word.transform(te["text"]), char.transform(te["text"]),
              onehot(te["target"])]).tocsr()

# ---- targets --------------------------------------------------------- #
Y = np.zeros((len(tr), len(pairs)))
aux = {k: [] for k in ("default_category", "top_competitor",
                       "target_reachable", "order_sensitive", "peak_bin")}
for i, r in enumerate(tr.itertuples(index=False)):
    m = surf[r.article_id]
    v = m[:, cat_i[r.target]]
    span = v.max() - v.min()
    Y[i] = (v - v.min()) / span if span > 0 else 0.0

    am = m.argmax(axis=1)
    aux["default_category"].append(cats[int(np.bincount(
        am, minlength=len(cats)).argmax())])
    col = m.mean(axis=0)
    aux["top_competitor"].append(
        cats[[j for j in np.argsort(-col) if j != cat_i[r.target]][0]])
    aux["target_reachable"].append(
        "yes" if (am == cat_i[r.target]).any() else "no")
    gaps = [abs(v[pair_i[(a, b)]] - v[pair_i[(b, a)]])
            for k, a in enumerate(comps) for b in comps[k + 1:]]
    aux["order_sensitive"].append(
        "yes" if span > 0 and np.mean(gaps) / span > 0.10 else "no")
    aux["peak_bin"].append(prob_bin(float(v.max())))

print(f"{len(tr)} train items, {len(te)} test items, "
      f"{len(surf)} measured surfaces")

# ---- fit ------------------------------------------------------------- #
ridge = Ridge(alpha=2.0, random_state=SEED)
ridge.fit(Xtr, Y)
Yhat = ridge.predict(Xte)

heads = {}
for k, labels in aux.items():
    labels = np.asarray(labels)
    if len(set(labels)) < 2:
        heads[k] = ("const", labels[0])
        continue
    clf = LogisticRegression(max_iter=2000, C=4.0, random_state=SEED)
    clf.fit(Xtr, labels)
    heads[k] = ("model", clf.predict(Xte))

def head(k, i):
    kind, val = heads[k]
    return val if kind == "const" else val[i]

# ---- read every field off the predicted surface ---------------------- #
out = []
for i, r in enumerate(te.itertuples(index=False)):
    v = Yhat[i]
    best = pairs[int(np.argmax(v))]

    first_mean, all_mean = {}, {}
    for c in comps:
        fi = [pair_i[(c, b)] for b in comps]
        si = [pair_i[(a, c)] for a in comps]
        first_mean[c] = v[fi].mean()
        all_mean[c] = v[sorted(set(fi + si))].mean()
    ranked = sorted(comps, key=lambda c: (-all_mean[c], c))

    rec = {
        "chosen_pair": {"first": best[0], "second": best[1]},
        "best_first_component": min(comps,
                                    key=lambda c: (-first_mean[c], c)),
        "helpful_components": sorted(ranked[:TOP_K]),
        "harmful_components": sorted(ranked[-TOP_K:]),
        "default_category": str(head("default_category", i)),
        "top_competitor": str(head("top_competitor", i)),
        "target_reachable": str(head("target_reachable", i)),
        "order_sensitive": str(head("order_sensitive", i)),
        "peak_probability_bin": str(head("peak_bin", i)),
    }
    out.append({"id": r.id, "answer_json": json.dumps(rec,
                                                      ensure_ascii=False)})

work = os.path.join(here, "working")
os.makedirs(work, exist_ok=True)
pd.DataFrame(out).to_csv(os.path.join(work, "submission.csv"), index=False)
print(f"wrote {len(out)} rows to working/submission.csv")

## 5. Notes on what matters

- Validate by **article**, never by row. Each article contributes three items;
  splitting rows at random leaks an article's whole surface into fit.
- Keep the pair **ordered**. (C04, C12) and (C12, C04) are different prompts
  and generally different measurements.
- The 256 pairs are an outer product of a 16-element catalog, so the surface is
  close to low rank: a first-position effect, a second-position effect and an
  interaction. Factorising it that way uses the training articles more
  efficiently than treating the 256 outputs as unrelated.